In [2]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
import time
import random

In [3]:
def check_volcano_exists(volcano_id):
    """
    Verifica si un ID de volcán existe en el Global Volcanism Program
    """
    url = f"https://volcano.si.edu/volcano.cfm?vn={volcano_id}"
    try:
        response = requests.get(url, timeout=10)
        
        # Condiciones para determinar si el volcán existe
        # Estas condiciones pueden necesitar ajuste según la estructura real de la página
        if response.status_code == 200 and len(response.text) > 5000:
            return {
                'id': volcano_id, 
                'url': url
            }
        return None
    except requests.RequestException:
        return None

In [4]:
def extract_volcano_ids(start_id, end_id, max_workers=20):
    """
    Extrae IDs de volcanes en un rango específico usando concurrencia
    """
    existing_volcanoes = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Generar lista de IDs a verificar
        futures = {
            executor.submit(check_volcano_exists, volcano_id): volcano_id 
            for volcano_id in range(start_id, end_id + 1)
        }
        
        for future in as_completed(futures):
            result = future.result()
            if result:
                existing_volcanoes.append(result)
                print(f"Encontrado: {result['id']}")
            
            # Para no sobrecargar el servidor
            time.sleep(random.uniform(0.1, 0.5))
    
    return existing_volcanoes


In [ ]:
existing_volcanoes = extract_volcano_ids(210010, 600000, max_workers=20)

Encontrado: 210011
Encontrado: 210023
Encontrado: 210030
Encontrado: 210022
Encontrado: 210034
Encontrado: 210032
Encontrado: 210040
Encontrado: 210043
Encontrado: 210035
Encontrado: 210047
Encontrado: 210042
Encontrado: 210031
Encontrado: 210041
Encontrado: 210046
Encontrado: 210033
Encontrado: 210044
Encontrado: 210036
Encontrado: 210045
Encontrado: 210039
Encontrado: 210037
Encontrado: 210038
Encontrado: 210048
Encontrado: 210050
Encontrado: 210053
Encontrado: 210051
Encontrado: 210054
Encontrado: 210060
Encontrado: 210059
Encontrado: 210065
Encontrado: 210064
Encontrado: 210068
Encontrado: 210070
Encontrado: 210071
Encontrado: 210072
Encontrado: 210073
Encontrado: 210074
Encontrado: 210075
Encontrado: 210076
Encontrado: 210077
Encontrado: 210078
Encontrado: 210079
Encontrado: 210080
Encontrado: 210081
Encontrado: 210082
Encontrado: 210083
Encontrado: 210049
Encontrado: 210084
Encontrado: 210085
Encontrado: 210086
Encontrado: 210087
Encontrado: 210062
Encontrado: 210088
Encontrado: 

In [5]:
def save_to_csv(volcanoes, filename='volcano_ids.csv'):
    """
    Guarda los IDs de volcanes encontrados en un archivo CSV
    """
    with open(filename, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['id', 'url'])
        writer.writeheader()
        writer.writerows(volcanoes)

In [6]:
def main():
    # Rango de IDs que mencionaste
    start_id = 210010
    end_id = 600000
    
    print(f"Buscando volcanes entre {start_id} y {end_id}")
    
    existing_volcanoes = extract_volcano_ids(start_id, end_id)
    
    print(f"Total de volcanes encontrados: {len(existing_volcanoes)}")
    
    save_to_csv(existing_volcanoes)
    print("IDs guardados en volcano_ids.csv")